<a href="https://colab.research.google.com/github/betmutema/newest/blob/main/w07_action_playbook_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This notebook turns the validated ranking output into a practical, human-reviewed content action playbook.

The model is treated as **decision support, not a decision**. The current honest validation result is precision@50 = **0.200**, below the **0.279** base rate on the client-held-out test. The queue therefore prioritises what a reviewer should inspect first; it does not decide which pages should be refreshed.

The notebook exports the ranked queue and metrics to `work/outputs/` for use by the research paper.

## 1. Ranked actions + reason codes

*The queue: what to review first, why it was surfaced, and what kind of review it needs.*

In [ ]:
import os, getpass, duckdb, pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
ANCHOR = "DATE '2026-03-31'"

features = con.sql(f'''
    SELECT f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date > {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
        AVG(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_prev30
    FROM {fact} f
    WHERE f.report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2
    HAVING imp_prev30 >= 100
''').df()

features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']

features = features.merge(
    con.sql(f'''
        SELECT content_hash_id,
               DATE_DIFF('day', content_created_date, {ANCHOR}) AS content_age_days
        FROM {dim_content}
    ''').df(),
    on='content_hash_id',
    how='left'
)

features['is_declining'] = (
    features['imp_last30'] < 0.8 * features['imp_prev30']
).astype(int)

features = features.dropna(
    subset=['pos_prev30', 'content_age_days']
).reset_index(drop=True)

FEATURE_COLS = [
    'imp_prev30', 'clk_prev30', 'pos_prev30',
    'ctr_prev30', 'content_age_days'
]

gss = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)

train_idx, test_idx = next(
    gss.split(features, groups=features['client_hash_id'])
)

train = features.iloc[train_idx].copy()
test = features.iloc[test_idx].copy()

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

rf.fit(train[FEATURE_COLS], train['is_declining'])
test['model_score'] = rf.predict_proba(test[FEATURE_COLS])[:, 1]

# Transparent review rule carried forward from ML-07.
stale = (test['content_age_days'] >= 365)
visible = (test['imp_prev30'] >= 250)

test['reason_code'] = np.where(
    stale & visible,
    'stale_but_visible',
    'model_flagged_only'
)

# Turn the reason code into an explicit review archetype and action.
test['archetype'] = np.where(
    test['reason_code'] == 'stale_but_visible',
    'refresh_candidate',
    'investigate_decline'
)

test['recommended_action'] = np.where(
    test['reason_code'] == 'stale_but_visible',
    'review_for_refresh',
    'review_underlying_signals'
)

# No confidence label is added because validation does not justify one.
test['review_priority'] = 'review'

queue = (
    test
    .sort_values('model_score', ascending=False)
    .reset_index(drop=True)
)

queue['rank'] = queue.index + 1

queue_cols = [
    'rank',
    'client_hash_id',
    'content_hash_id',
    'model_score',
    'reason_code',
    'archetype',
    'recommended_action',
    'review_priority',
    'content_age_days',
    'imp_prev30',
    'pos_prev30',
    'ctr_prev30'
]

print(queue[queue_cols].head(10).to_string(index=False))

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rank          client_hash_id          content_hash_id  model_score        reason_code           archetype        recommended_action review_priority  content_age_days  imp_prev30  pos_prev30  ctr_prev30
    1 client_62f4a7e64f5e0096 content_61d2ced86b23a928     0.657368 model_flagged_only investigate_decline review_underlying_signals          review               266       182.0   56.737646         0.0
    2 client_62f4a7e64f5e0096 content_f2c87dcda8cb44f7     0.657105 model_flagged_only investigate_decline review_underlying_signals          review               265       185.0   56.903993         0.0
    3 client_62f4a7e64f5e0096 content_998f37424ce64e55     0.655769 model_flagged_only investigate_decline review_underlying_signals          review               266       151.0   51.419452         0.0
    4 client_62f4a7e64f5e0096 content_1aee3c2049ed782d     0.655608 model_flagged_only investigate_decline review_underlying_signals          review               266       207.0   44.1210

### Archetype → action mapping

The model score ranks the queue, but the **reason code determines the type of human review**.

| Archetype | Trigger | Recommended action | Human check |
|---|---|---|---|
| `refresh_candidate` | Content age ≥ 365 days and prior-30-day impressions ≥ 250 | `review_for_refresh` | Check whether the content is actually outdated, whether search intent has changed, and whether the decline is meaningful |
| `investigate_decline` | Model flagged the page but it does not meet the transparent stale-and-visible rule | `review_underlying_signals` | Check impressions, position, CTR, content age and recent site/content changes |

These are **review categories**, not validated predictions. The model does not decide that a page needs refreshing.

In [ ]:
# Quick audit of the action mapping.
print(
    queue[
        ['reason_code', 'archetype', 'recommended_action']
    ].drop_duplicates().to_string(index=False)
)

top10_client_share = (
    queue.head(10)['client_hash_id'].value_counts(normalize=True).iloc[0]
)

print(f"Top-10 queue concentration in largest client: {top10_client_share:.1%}")

       reason_code           archetype        recommended_action
model_flagged_only investigate_decline review_underlying_signals
 stale_but_visible   refresh_candidate        review_for_refresh
Top-10 queue concentration in largest client: 100.0%


The current run's top 10 can be dominated by one client. That is a practical warning sign: a high model score does not necessarily mean the page is independently more important. Client concentration is therefore a reason to inspect the underlying features rather than blindly acting on the ranking.

### Decay / refresh insight

The earlier ML-07 signal check found:

| Content age | n | Observed decline rate |
|---|---:|---:|
| `<90d` | 16,071 | 16.5% |
| `90–365d` | 53,545 | 32.6% |
| `1–2yr` | 12,933 | 27.1% |

The relationship is **not monotonic**. Decline is higher for the 90–365 day group than for content under 90 days, but it falls again in the 1–2 year group.

Therefore, content age is useful as a **review eligibility signal**, not as proof that a page is declining. The `stale_but_visible` rule should be read as “worth checking” rather than “needs refreshing.”

The rule also combines age with existing visibility because visibility provides a practical prioritisation signal: if a page does need a refresh, an already-visible page may have more potential search impact. That does not mean higher visibility makes decline more likely.

## 2. Intended use and limits

*Who uses this, what it is for, and where the evidence stops.*

**Intended use:** this is a starting shortlist for a content strategist or SEO editor who has limited review time. The queue identifies pages to inspect first and gives each page a reason code and review action.

**Decision boundary:** the model can help prioritise review, but it cannot make the final refresh/no-refresh decision.

**Main validation limit:** on a held-out set of clients that were not seen during training, precision@50 was **0.200**, below the **0.279** base rate. The model therefore has not been shown to outperform random selection at this threshold.

The earlier naive split produced a much higher precision@50, but client overlap made that result unreliable for judging generalisation. The client-grouped result is the number used for this playbook.

The model score should therefore be treated as a ranking signal, not as a calibrated probability that a page needs a refresh.

The transparent `stale_but_visible` rule remains useful because its conditions can be checked directly: content age ≥ 365 days and prior-30-day impressions ≥ 250.

## 3. Human review + the no-go list

*What a person must check before acting, and what must never be automated.*

**Required human review:** every flagged item must be reviewed before any action is taken. The model has not cleared the base rate, so treating its output as pre-validated would be misleading.

### Human review checklist

For each queued page, the reviewer should check:

1. Is the page actually declining according to recent search performance?
2. Is the content outdated or no longer aligned with current search intent?
3. Has the page recently been changed, migrated or affected by another site change?
4. Is the observed movement large enough to justify editorial work?
5. Does the page have enough existing visibility that a successful refresh could have meaningful value?
6. Is there a clear reason to refresh rather than leave the page unchanged?

### What must NOT be automated

- Do not automatically publish rewritten content.
- Do not automatically rewrite or delete a page.
- Do not automatically refresh every page above a model-score threshold.
- Do not automatically reduce marketing or development budgets because a page is labelled declining.
- Do not present the model score to a client as a certainty metric.
- Do not act on `model_flagged_only` items without checking the underlying feature values.
- Do not treat content age alone as proof that a page needs refreshing.
- Do not turn the queue into a production workflow without fresh validation.

### Cost / value thinking

This notebook does **not** estimate financial ROI because the dataset does not contain measured editorial labour cost, expected traffic gain from a refresh, revenue impact, or opportunity cost.

The practical trade-off is therefore qualitative:

- **Potential value:** existing search visibility is a simple proxy for how much search exposure a successful refresh could affect.
- **Review cost:** every queued item consumes human editorial time.
- **Decision:** prioritise human review where there is a plausible refresh signal and meaningful existing visibility, but do not claim that the queue estimates financial return.

The earlier visibility analysis also showed that higher impression volume was not itself evidence of higher decline risk. Visibility is therefore a value/prioritisation signal, not a prediction of decline.

## 4. Monitoring / retrain triggers

*What would tell us that the playbook has gone stale, and when retraining is justified.*

### Monitoring

- Re-run precision@50 against the base rate quarterly on a fresh, client-held-out evaluation set.
- Monitor the decline base rate. If it changes materially, re-establish the comparison point before interpreting precision@50.
- Monitor the distribution of `content_age_days`, `imp_prev30`, `ctr_prev30` and `pos_prev30`. Large shifts mean the current ranking may no longer represent the data being reviewed.
- Monitor client concentration in the top-ranked queue. If one client repeatedly dominates the top of the queue, inspect whether client-specific scale is driving the ranking.

### Retrain / re-validation triggers

- A substantial change in the client roster should trigger fresh validation before trusting the ranking.
- A material feature-distribution shift should trigger investigation and re-validation.
- A sustained drop in precision@50 relative to the base rate should trigger investigation.
- Retraining should **not** be automatic. First check whether the task definition, features, target construction and validation design are still appropriate.
- Retrain only when new labelled data is available and fresh client-held-out validation shows that the revised model improves the decision-support use case.

## 5. Exports for the paper

*Generate the exact queue and metrics files that the next research-paper stage can reuse.*

In [ ]:
import os
import json

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# The queue stays out of git by design; the notebook regenerates it.
queue.to_csv(
    'work/outputs/content_action_playbook_queue.csv',
    index=False
)

metrics = {
    'precision_at_50': 0.200,
    'base_rate': 0.279,
    'naive_split_precision_at_50': 0.92,
    'grouped_split_precision_at_50': 0.20,
    'validation_design': 'client_grouped_split',
    'decision_use': 'decision_support_only',
    'note': (
        'Model did not exceed the base rate on the grouped/client-held-out split. '
        'Every queued item requires human review.'
    )
}

with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported:")
print(" - work/outputs/content_action_playbook_queue.csv")
print(" - work/outputs/playbook_metrics.json")

Exported:
 - work/outputs/content_action_playbook_queue.csv
 - work/outputs/playbook_metrics.json


The exported queue now contains the fields needed by the paper:

- `rank`
- `model_score`
- `reason_code`
- `archetype`
- `recommended_action`
- `review_priority`
- the underlying review features needed for human inspection

The paper should describe this as a **human-reviewed action queue**, not an automated refresh system.

## Self-check

Before submitting, confirm each line honestly:

- [x] Ranked queue exists with reason codes.
- [x] Archetype → action mapping is explicit.
- [x] Decay/refresh insight is included and does not overclaim causality.
- [x] Intended use and model limits are explicit.
- [x] Human review rules are explicit.
- [x] No-go automation cases are explicit.
- [x] Cost/value thinking is included without inventing financial numbers.
- [x] Monitoring and retrain triggers are practical and non-production.
- [x] Queue and metrics are exported to `work/outputs/`.
- [ ] Run the notebook top-to-bottom in Colab with the project `HF_TOKEN`, then commit the executed notebook.
- [ ] Confirm the generated queue and metrics files exist locally; keep the queue out of git if the repository's leak-guard requires it.
- [ ] Commit the corrected notebook under `work/notebooks/w07_action_playbook.ipynb`.